# Main Figures 2 and 3: robust spectroscopy maps and spectral slices

This notebook uses matched $L=20\,\mu\mathrm{s}$ simulations to compare constant, root-Lorentzian, and echo-root-Lorentzian spectroscopy over 50-MHz-wide and 1-MHz-wide detuning domains. Figure 2 shows the two-dimensional maps. Figure 3 compares fixed-amplitude root-Lorentzian and echo-root-Lorentzian spectral slices at three peak Rabi frequencies.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'echospec').is_dir())
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.codex_tmp' / 'mpl'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from echospec.figures import FigureVariant, apply_figure_style, save_figure
import scripts.make_simulated_echo_lorentzian_duration_cutoff_comparison as supplemental_simulation

VARIANT = FigureVariant.PAPER
SOURCE_PATH = PROJECT_ROOT / 'figures/paper/09_simulated_echo_lorentzian_20us.npz'
CUTOFF = 0.002
BROAD_SPAN_MHZ = 50.0
NARROW_SPAN_MHZ = 1.0
apply_figure_style(VARIANT)
print(SOURCE_PATH)

/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/09_simulated_echo_lorentzian_20us.npz


In [2]:
with np.load(SOURCE_PATH) as archived:
    duration_us = float(archived['duration_us'])
    cutoffs = np.asarray(archived['cutoffs'], dtype=float)
    detuning_mhz = np.asarray(archived['detuning_mhz'], dtype=float)
    rabi_mhz = np.asarray(archived['rabi_mhz'], dtype=float)
    t1_us = float(archived['t1_us'])
    t_phi_us = float(archived['t_phi_us'])
    archived_maps = {
        'lorentzian': np.asarray(archived[f'lorentzian_cutoff_{CUTOFF:g}'], dtype=float),
        'echo': np.asarray(archived[f'echo_cutoff_{CUTOFF:g}'], dtype=float),
    }

broad_detuning_mhz = np.linspace(-BROAD_SPAN_MHZ / 2, BROAD_SPAN_MHZ / 2, 401)
narrow_detuning_mhz = np.linspace(-NARROW_SPAN_MHZ / 2, NARROW_SPAN_MHZ / 2, 401)
supplemental_simulation.RABI_MHZ = rabi_mhz

def simulate_protocol_pair(detuning_grid_mhz, *, steps_per_half):
    supplemental_simulation.DETUNING_MHZ = detuning_grid_mhz
    supplemental_simulation.U_STEPS_PER_HALF = steps_per_half
    return {
        'lorentzian': supplemental_simulation.simulate_map(duration_us, CUTOFF, echo=False),
        'echo': supplemental_simulation.simulate_map(duration_us, CUTOFF, echo=True),
    }

broad_maps = simulate_protocol_pair(broad_detuning_mhz, steps_per_half=6400)
narrow_maps = simulate_protocol_pair(narrow_detuning_mhz, steps_per_half=1600)

def simulate_constant_map(detuning_grid_mhz):
    detuning_rad_us, rabi_rad_us = np.meshgrid(
        2 * np.pi * detuning_grid_mhz, 2 * np.pi * rabi_mhz
    )
    t2_us = 1 / (1 / (2 * t1_us) + 1 / t_phi_us)
    generator = np.zeros(detuning_rad_us.shape + (4, 4), dtype=float)
    generator[..., 0, 0] = -1 / t2_us
    generator[..., 0, 1] = detuning_rad_us
    generator[..., 1, 0] = -detuning_rad_us
    generator[..., 1, 1] = -1 / t2_us
    generator[..., 1, 2] = -rabi_rad_us
    generator[..., 2, 1] = rabi_rad_us
    generator[..., 2, 2] = -1 / t1_us
    generator[..., 2, 3] = 1 / t1_us
    initial_state = np.asarray([0.0, 0.0, 1.0, 1.0])
    final_state = np.einsum(
        '...ij,j->...i', expm(generator * duration_us), initial_state
    )
    population = (1 - final_state[..., 2]) / 2
    if not np.isfinite(population).all():
        raise RuntimeError('Constant-pulse simulation produced nonfinite values')
    if population.min() < -1e-10 or population.max() > 1 + 1e-10:
        raise RuntimeError('Constant-pulse simulation left the physical range')
    return np.clip(population, 0, 1)

broad_maps['constant'] = simulate_constant_map(broad_detuning_mhz)
narrow_maps['constant'] = simulate_constant_map(narrow_detuning_mhz)

assert np.isclose(duration_us, 20.0)
assert np.any(np.isclose(cutoffs, CUTOFF))
assert np.isclose(t1_us, supplemental_simulation.T1_US)
assert np.isclose(t_phi_us, supplemental_simulation.T_PHI_US)
for protocol, archived_population in archived_maps.items():
    assert archived_population.shape == (rabi_mhz.size, detuning_mhz.size), protocol
    assert broad_maps[protocol].shape == (rabi_mhz.size, broad_detuning_mhz.size)
    assert narrow_maps[protocol].shape == (rabi_mhz.size, narrow_detuning_mhz.size)
    assert np.isfinite(broad_maps[protocol]).all(), protocol
    assert np.isfinite(narrow_maps[protocol]).all(), protocol
    assert broad_maps[protocol].min() >= 0 and broad_maps[protocol].max() <= 1
    assert narrow_maps[protocol].min() >= 0 and narrow_maps[protocol].max() <= 1
    archived_center = archived_population[:, np.abs(detuning_mhz) <= NARROW_SPAN_MHZ / 2 + 1e-12]
    np.testing.assert_allclose(narrow_maps[protocol][:, ::8], archived_center, atol=2e-12)
for population in (broad_maps['constant'], narrow_maps['constant']):
    assert population.shape[0] == rabi_mhz.size
    assert np.isfinite(population).all()
    assert population.min() >= 0 and population.max() <= 1

effective_t2_us = 1 / (1 / (2 * t1_us) + 1 / t_phi_us)
print(
    f'L={duration_us:g} us, c={CUTOFF:g}, T1={t1_us:.2f} us, '
    f'Tphi={t_phi_us:.2f} us, T2_eff={effective_t2_us:.3f} us'
)
print(f'Broad detuning grid: {broad_detuning_mhz.size} points from {broad_detuning_mhz.min():g} to {broad_detuning_mhz.max():g} MHz')
print(f'Narrow detuning grid: {narrow_detuning_mhz.size} points from {narrow_detuning_mhz.min():g} to {narrow_detuning_mhz.max():g} MHz')
print(f'Peak-Rabi grid: {rabi_mhz.size} points from {rabi_mhz.min():g} to {rabi_mhz.max():g} MHz')

L=20 us, c=0.002, T1=51.24 us, Tphi=7.87 us, T2_eff=7.309 us
Broad detuning grid: 401 points from -25 to 25 MHz
Narrow detuning grid: 401 points from -0.5 to 0.5 MHz
Peak-Rabi grid: 81 points from 0 to 50 MHz


In [3]:
plt.rcParams.update({
    'figure.figsize': (7.1, 3.65),
    'axes.titlesize': 7.0,
    'axes.labelsize': 6.8,
    'xtick.labelsize': 6.2,
    'ytick.labelsize': 6.2,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

fig, axes = plt.subplots(2, 3, sharey=True, constrained_layout=True)
protocols = (
    ('constant', 'Constant'),
    ('lorentzian', 'Root-Lorentzian'),
    ('echo', 'Echo-root-Lorentzian'),
)
views = (
    ('Broad domain', BROAD_SPAN_MHZ, broad_detuning_mhz, broad_maps),
    ('Narrow domain', NARROW_SPAN_MHZ, narrow_detuning_mhz, narrow_maps),
)
panel_labels = iter('abcdef')
image = None

for row, (view_label, span_mhz, plot_detuning, plot_maps) in enumerate(views):
    for column, (protocol, protocol_label) in enumerate(protocols):
        ax = axes[row, column]
        plot_population = plot_maps[protocol]
        image = ax.pcolormesh(
            plot_detuning,
            rabi_mhz,
            plot_population,
            shading='auto',
            cmap='viridis',
            vmin=0.0,
            vmax=0.5,
            rasterized=True,
        )
        ax.set_xlim(-span_mhz / 2, span_mhz / 2)
        ax.set_ylim(rabi_mhz.min(), rabi_mhz.max())
        ax.set_box_aspect(0.72)
        ax.axvline(0.0, color='white', lw=0.55, ls='--', alpha=0.85)
        ax.text(
            0.035, 0.95, f'({next(panel_labels)})',
            transform=ax.transAxes, ha='left', va='top',
            color='white', fontweight='bold',
        )
        if row == 0:
            ax.set_title(protocol_label)
        if row == 1:
            ax.set_xlabel(r'Detuning, $\Delta/2\pi$ (MHz)')
        if column == 0:
            ax.set_ylabel(view_label + rf' ({span_mhz:g} MHz span)' + '\n' + r'$\Omega_0/2\pi$ (MHz)')

if image is None:
    raise RuntimeError('No panels were generated')
axes[0, 0].text(
    0.96, 0.95, rf'$L={duration_us:g}\,\mu\mathrm{{s}}$',
    transform=axes[0, 0].transAxes, ha='right', va='top',
    color='white', fontsize=6.0,
)
axes[0, 1].text(
    0.96, 0.95, rf'$c={CUTOFF:g}$',
    transform=axes[0, 1].transAxes, ha='right', va='top',
    color='white', fontsize=6.0,
)
colorbar = fig.colorbar(image, ax=axes, pad=0.018, fraction=0.05)
colorbar.set_label(r'Final excited-state probability, $P_e$')
colorbar.ax.tick_params(labelsize=6.0)

saved = save_figure(
    fig,
    '02_central_spectroscopy',
    variant=VARIANT,
    formats=('pdf', 'png', 'svg'),
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.04,
)
saved

[PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/02_central_spectroscopy.pdf'),
 PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/02_central_spectroscopy.png'),
 PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/02_central_spectroscopy.svg')]

In [4]:
# Main-text Fig. 3: matched spectral slices at three drive amplitudes.
TARGET_SLICE_RABI_MHZ = np.asarray([2.5, 10.0, 40.0])
slice_indices = np.asarray([
    int(np.argmin(np.abs(rabi_mhz - target)))
    for target in TARGET_SLICE_RABI_MHZ
])
selected_slice_rabi_mhz = rabi_mhz[slice_indices]
np.testing.assert_allclose(selected_slice_rabi_mhz, TARGET_SLICE_RABI_MHZ)

slice_protocols = (
    ('lorentzian', 'Root-Lorentzian', '#00838f'),
    ('echo', 'Echo-root-Lorentzian', '#6a1b9a'),
)
fig_slices, slice_axes = plt.subplots(
    3, 2, figsize=(3.38, 4.45), sharex=True, sharey=True,
    constrained_layout=True,
)
slice_panel_labels = iter('abcdef')

for row, (amplitude_index, amplitude_mhz) in enumerate(
    zip(slice_indices, selected_slice_rabi_mhz)
):
    for column, (protocol, title, color) in enumerate(slice_protocols):
        ax = slice_axes[row, column]
        ax.plot(
            detuning_mhz, archived_maps[protocol][amplitude_index],
            color=color, lw=1.25,
        )
        ax.axvline(0.0, color='0.45', lw=0.55, ls='--', zorder=0)
        ax.set_xlim(detuning_mhz.min(), detuning_mhz.max())
        ax.set_ylim(0.0, 0.82)
        ax.set_box_aspect(1.0)
        ax.text(
            0.04, 0.94, f'({next(slice_panel_labels)})',
            transform=ax.transAxes, ha='left', va='top',
            fontweight='bold',
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.78, 'pad': 0.5},
        )
        if row == 0:
            ax.set_title(title)
        if row == 2:
            ax.set_xlabel(r'Detuning, $\Delta/2\pi$ (MHz)')
        if column == 0:
            ax.set_ylabel(
                rf'$P_e$' + '\n' + rf'$\Omega_0/2\pi={amplitude_mhz:g}$ MHz'
            )

saved_slices = save_figure(
    fig_slices,
    '03_lorentzian_echo_slices',
    variant=VARIANT,
    formats=('pdf', 'png', 'svg'),
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.04,
)
print('Selected peak-Rabi frequencies (MHz):', selected_slice_rabi_mhz)
saved_slices

Selected peak-Rabi frequencies (MHz): [ 2.5 10.  40. ]


[PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/03_lorentzian_echo_slices.pdf'),
 PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/03_lorentzian_echo_slices.png'),
 PosixPath('/Users/asafsolonnikov/Developer/amplitude-robust-spectroscopy/figures/paper/03_lorentzian_echo_slices.svg')]

### Provenance

The Lorentzian maps and spectral slices are calculated with the identical Supplemental generator and parameter set. The broad calculation uses 6400 integration steps per pulse half for stability at $|\Delta|/2\pi=25$ MHz; the narrow calculation retains the archived 1600-step setting and is numerically verified against every overlapping point in the archived Supplemental arrays. The constant-pulse row is evaluated exactly from the time-independent affine Bloch generator using a batched matrix exponential. No interpolation, normalization, or independent color rescaling is applied. All protocols use the measurement-linked $T_1$ and conservative effective transverse-coherence parameters recorded in the archive.